# Orientation Experiments 6–9

Tests four approaches that avoid the pixel-aligned polygon problem:

- **Baseline**: standard binary mask → `binary_mask_to_polygon_cv` → `segmentize` → `fitEllipse`
- **Exp 6**: SAM2 logit isocontour — `skimage.find_contours(logit, 0.0)` → sub-pixel smooth polygon → `fitEllipse`
- **Exp 7**: Canny edges on the image patch inside the mask → `cv2.fitEllipse` on edge pixel coords
- **Exp 8**: Soft-mask image moments — logit-weighted `(mu11, mu20, mu02)` → orientation angle
- **Exp 9**: SAM2 re-prompted with a point (mask centroid) instead of a bounding box

All run on the same 5 Prieur et al. ground-truth test tiles used in Exp 3b.
Success criterion: the 90°/175° spikes in `angle180` flatten out.

**Key difference from all previous experiments**: Exps 6 and 8 use the SAM2 *logit map* (raw float output before thresholding) instead of the binary mask. This requires capturing the 3rd return value of `predict_batch`.

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch

from pathlib import Path
from PIL import Image
from tqdm import tqdm
from shapely.geometry import Polygon
from shapely import segmentize
from skimage.measure import find_contours

from sahi import AutoDetectionModel
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

from rastertools_BOULDERING import convert as raster_convert, metadata as raster_metadata
from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

In [ ]:
# ── GPU availability check ────────────────────────────────────────────────
# If you hit 'CUDA-capable device is busy or unavailable':
#   1. Kernel → Restart Kernel (kills any lingering CUDA context in THIS kernel)
#   2. Run `! nvidia-smi` to see which PIDs are holding the device
#   3. On Sherlock: confirm your srun/sbatch includes --gres=gpu:1
#      and that no other open Jupyter kernels are using the same GPU
torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device available.\n"
        "  - On Sherlock: make sure your srun/sbatch uses --gres=gpu:1\n"
        "  - Shut down other open notebook kernels: Kernel → Shut Down All Kernels\n"
        "  - Run  ! nvidia-smi  to see which PIDs are using the GPU")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}  ({gpu_mem:.1f} GB)")
print(f"CUDA version: {torch.version.cuda}")

# ── Paths ─────────────────────────────────────────────────────────────────
home_p          = Path.home()
work_dir        = home_p / "tmp" / "YOLOv8BeyondEarth"
prieur_dir      = Path("/scratch/users/cayleigh/Apr2023-Mars-Moon-Earth-mask-5px")
prieur_test_dir = prieur_dir / "preprocessing" / "test"
tile_tmp_dir    = work_dir / "tile_tmp"
tile_tmp_dir.mkdir(parents=True, exist_ok=True)

gt_tile_ids = ["1386", "1503", "2054", "2277", "2508"]

model_weights   = work_dir / "yolov8_model" / "yolov8-m-boulder-detection-tmp.pt"
sam2_checkpoint = Path("/scratch/users/cayleigh/checkpoints/sam2.1_hiera_small.pt")

# ── Models ────────────────────────────────────────────────────────────────
torch.cuda.empty_cache()
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=model_weights.as_posix(),
    confidence_threshold=0.1,
    device="cuda:0",
    image_size=1024)

sam2_base_model = build_sam2(
    "configs/sam2.1/sam2.1_hiera_s.yaml", sam2_checkpoint, device="cuda:0")
predictor = SAM2ImagePredictor(sam2_base_model)

# ── Tile resolution ───────────────────────────────────────────────────────
tile_res = raster_metadata.get_resolution(
    prieur_test_dir / "images" / f"M1221383405_{gt_tile_ids[0]}_image.tif"
)[0]
print(f"Tile resolution: {tile_res:.4f} m/px")

## Data collection

Run YOLO + SAM2 on each test tile and save raw per-detection data:
- `box_mask` (256×256 bool): standard binary mask from box-prompted SAM2
- `box_logit` (256×256 float): raw SAM2 logit map before thresholding
- `point_mask` (256×256 bool): mask from point-prompted SAM2 (centroid of `box_mask`)
- `tile_image` (H×W×3 uint8): the full tile (for gradient-based orientation)
- `bbox` [x1,y1,x2,y2]: YOLO detection in tile pixel coords

In [ ]:
CONFIDENCE_THRESHOLD = 0.10
MIN_AREA_THRESHOLD   = 6


def load_tile_rgb(tile_tif: Path, tile_tmp: Path) -> np.ndarray:
    """Return the tile as uint8 RGB numpy array (H, W, 3), converting TIF→PNG if needed."""
    tile_png = tile_tmp / f"{tile_tif.stem}.png"
    if not tile_png.exists():
        raster_convert.tiff_to_png(tile_tif, tile_png)
    img = Image.open(tile_png).convert("RGB")
    return np.array(img)


def collect_tile_detections(tile_image_np, detection_model, predictor,
                             confidence_threshold=0.10, min_area_threshold=6):
    """
    Run YOLO + SAM2 on a single tile (no slicing).
    Returns list of per-detection dicts with raw masks and logits.

    predict_batch returns (masks, iou_scores, low_res_logits):
      masks[j]:      (N_boxes, 1, H_img, W_img) bool
      low_res_logits[j]: (N_boxes, 1, 256, 256)  float  ← raw SAM2 logits
    """
    with torch.no_grad():
        yolo_results = detection_model.model(
            [tile_image_np], imgsz=detection_model.image_size,
            verbose=False, device=detection_model.device)

    boxes_data = yolo_results[0].boxes.data
    conf_mask  = boxes_data[:, 4] >= confidence_threshold
    boxes_data = boxes_data[conf_mask]

    if len(boxes_data) == 0:
        return []

    bboxes    = boxes_data[:, :4].cpu().numpy()  # (N, 4)
    scores    = boxes_data[:, 4].cpu().numpy()
    categories = boxes_data[:, 5].cpu().numpy()

    # Box-prompted SAM2 — captures logit maps for Exp 6 and 8
    with torch.no_grad():
        predictor.set_image_batch([tile_image_np])
        box_masks_batch, _, box_logits_batch = predictor.predict_batch(
            None, None,
            box_batch=[bboxes],
            multimask_output=False)
    # box_masks_batch[0]:  (N, 1, H, W) bool
    # box_logits_batch[0]: (N, 1, 256, 256) float
    box_masks  = box_masks_batch[0][:, 0]   # (N, H, W)
    box_logits = box_logits_batch[0][:, 0]  # (N, 256, 256)

    # Point-prompted SAM2 — use centroid of box_mask as prompt (Exp 9)
    H, W = tile_image_np.shape[:2]
    point_masks = np.zeros((len(bboxes), H, W), dtype=bool)

    with torch.no_grad():
        predictor.set_image(tile_image_np)
        for i in range(len(bboxes)):
            mask_yx = np.argwhere(box_masks[i])
            if len(mask_yx) > 0:
                cy, cx = mask_yx.mean(axis=0)
            else:
                x1, y1, x2, y2 = bboxes[i]
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            pt_masks, _, _ = predictor.predict(
                point_coords=np.array([[cx, cy]]),
                point_labels=np.array([1]),
                multimask_output=False)
            point_masks[i] = pt_masks[0]

    records = []
    for i in range(len(bboxes)):
        area = int(box_masks[i].sum())
        if area <= min_area_threshold:
            continue
        records.append({
            'bbox':        bboxes[i],
            'score':       float(scores[i]),
            'box_mask':    box_masks[i],    # (H, W) bool
            'box_logit':   box_logits[i],   # (256, 256) float  ← raw logit
            'point_mask':  point_masks[i],  # (H, W) bool
            'tile_image':  tile_image_np,   # (H, W, 3) uint8
            'poly_area':   area * (tile_res ** 2),
        })
    return records

In [ ]:
all_records = []

for tile_id in gt_tile_ids:
    tile_tif = prieur_test_dir / "images" / f"M1221383405_{tile_id}_image.tif"
    tile_image_np = load_tile_rgb(tile_tif, tile_tmp_dir)
    recs = collect_tile_detections(
        tile_image_np, detection_model, predictor,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        min_area_threshold=MIN_AREA_THRESHOLD)
    for r in recs:
        r['tile_id'] = tile_id
    all_records.extend(recs)
    print(f"tile {tile_id}: {len(recs)} detections")

print(f"\nTotal detections: {len(all_records)}")

## Orientation estimators

Each function takes a detection record and returns `{theta_deg, angle180, aspect_ratio}` or `None` on failure.

- `theta_deg`: angle of the major axis from `fitEllipse` (or equivalent), used to check for the 90° spike
- `angle180`: geographic orientation (CW from north, [0°, 180°)), used to check for the 175° spike
- `aspect_ratio`: a/b ratio — filter to 1.2–2.0 for elongated boulders

In [ ]:
def poly_to_orient(poly, res):
    """Shared downstream: segmentize → fitEllipse → boulder_row → (theta_deg, angle180, aspect_ratio)."""
    row_seg = pd.Series({"geometry": segmentize(poly, res)})
    ellipse_poly, a_fit, b_fit, theta_rad = fitEllipse(row_seg)
    theta_deg = np.degrees(theta_rad)
    mrr_row   = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
    vals      = boulder_row(mrr_row)
    long_axis, short_axis, angle180 = vals[2], vals[3], vals[7]
    aspect_ratio = long_axis / short_axis if short_axis > 0 else None
    return theta_deg, angle180, aspect_ratio


def binary_mask_to_poly(mask_bool):
    """CHAIN_APPROX_NONE contour of a bool mask → Shapely Polygon, or None."""
    mask_u8 = mask_bool.astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    pts = max(contours, key=cv2.contourArea).squeeze(1)
    if len(pts) < 4:
        return None
    return Polygon(pts.astype(float))


# ── Baseline ──────────────────────────────────────────────────────────────
def orient_baseline(rec, res=tile_res):
    poly = binary_mask_to_poly(rec['box_mask'])
    if poly is None:
        return None
    return poly_to_orient(poly, res)


# ── Exp 6: SAM2 logit isocontour ──────────────────────────────────────────
def orient_exp6(rec, res=tile_res):
    logit_2d    = rec['box_logit'].astype(float)
    contours_rc = find_contours(logit_2d, level=0.0)
    if not contours_rc:
        return None
    contour_rc = max(contours_rc, key=len)
    if len(contour_rc) < 5:
        return None
    contour_xy = contour_rc[:, ::-1]          # (row,col) → (x,y)
    poly = Polygon(contour_xy)
    if not poly.is_valid or poly.is_empty:
        return None
    return poly_to_orient(poly, res)


# ── Exp 7: Canny edge pixels → cv2.fitEllipse ─────────────────────────────
# Fix: lower Canny thresholds (10/30 instead of 20/60) + larger dilation
# to recover the ~60% of boulders that had too few edges at the old settings.
def orient_exp7(rec):
    tile_gray = cv2.cvtColor(rec['tile_image'], cv2.COLOR_RGB2GRAY)
    mask_u8   = rec['box_mask'].astype(np.uint8)

    # Dilate mask further so boundary edges aren't clipped
    dilated_mask = cv2.dilate(mask_u8, np.ones((3, 3), np.uint8), iterations=4)
    masked_gray  = cv2.bitwise_and(tile_gray, tile_gray, mask=dilated_mask)

    # Lower thresholds to catch low-contrast boulder edges
    edges    = cv2.Canny(masked_gray, threshold1=10, threshold2=30)
    edge_pts = np.argwhere(edges)             # (N, 2) in (row, col)

    if len(edge_pts) < 5:
        return None

    edge_xy = edge_pts[:, ::-1].astype(np.float32).reshape(-1, 1, 2)
    try:
        (cx, cy), (MA, ma), angle_deg = cv2.fitEllipse(edge_xy)
    except cv2.error:
        return None

    if ma <= 0:
        return None

    aspect_ratio = max(MA, ma) / min(MA, ma)
    theta_deg    = (90.0 - angle_deg) % 180
    angle180     = angle_deg % 180
    return theta_deg, angle180, aspect_ratio


# ── Exp 8: Soft-mask image moments ────────────────────────────────────────
# Fix: use max(logit, 0) as weights instead of sigmoid(logit).
# sigmoid gives nonzero weight to background pixels, making the moment
# matrix nearly circular → almost all aspect_ratios fall below 1.2 → n=1.
# Clamping negatives to 0 means only foreground-confident pixels contribute.
def orient_exp8(rec):
    logit_2d = rec['box_logit'].astype(float)   # (256, 256)
    x1, y1, x2, y2 = rec['bbox'].astype(int)
    H, W = logit_2d.shape
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(W, x2), min(H, y2)
    if x2 <= x1 or y2 <= y1:
        return None

    logit_crop = logit_2d[y1:y2, x1:x2]
    weights    = np.maximum(logit_crop, 0.0)    # only foreground pixels (logit > 0)
    total      = weights.sum()
    if total < 1e-6:
        return None

    rows, cols = np.indices(weights.shape)
    cx_local = (weights * cols).sum() / total
    cy_local = (weights * rows).sum() / total

    dx   = cols - cx_local
    dy   = rows - cy_local
    mu20 = (weights * dx**2).sum() / total
    mu02 = (weights * dy**2).sum() / total
    mu11 = (weights * dx * dy).sum() / total

    theta_rad = 0.5 * np.arctan2(2.0 * mu11, mu20 - mu02)
    theta_deg = np.degrees(theta_rad)
    angle180  = (90.0 - theta_deg) % 180

    trace = mu20 + mu02
    det   = mu20 * mu02 - mu11**2
    disc  = max(0.0, (trace / 2)**2 - det)
    lam_max = trace / 2 + np.sqrt(disc)
    lam_min = trace / 2 - np.sqrt(disc)
    aspect_ratio = np.sqrt(lam_max / lam_min) if lam_min > 1e-9 else None

    return theta_deg, angle180, aspect_ratio


# ── Exp 9: Point-prompted SAM2 ────────────────────────────────────────────
def orient_exp9(rec, res=tile_res):
    poly = binary_mask_to_poly(rec['point_mask'])
    if poly is None:
        return None
    if int(rec['point_mask'].sum()) <= MIN_AREA_THRESHOLD:
        return None
    return poly_to_orient(poly, res)


print("Orientation estimator functions defined.")

In [ ]:
estimators = [
    ("Baseline (binary mask)",    orient_baseline),
    ("Exp 6 (logit isocontour)",  orient_exp6),
    ("Exp 7 (Canny fitEllipse)",  orient_exp7),
    ("Exp 8 (soft moments)",      orient_exp8),
    ("Exp 9 (point prompt)",      orient_exp9),
]

orient_dfs = {}

for name, fn in estimators:
    records_out = []
    for rec in tqdm(all_records, desc=name):
        try:
            result = fn(rec)
        except Exception:
            result = None
        if result is None:
            records_out.append({'theta_deg': None, 'angle180': None,
                                 'aspect_ratio': None, 'poly_area': rec['poly_area']})
        else:
            theta_deg, angle180, aspect_ratio = result
            records_out.append({'theta_deg': theta_deg, 'angle180': angle180,
                                 'aspect_ratio': aspect_ratio, 'poly_area': rec['poly_area']})
    orient_dfs[name] = pd.DataFrame(records_out)

for name, df in orient_dfs.items():
    n_elong = ((df['aspect_ratio'] >= 1.2) & (df['aspect_ratio'] <= 2.0)).sum()
    print(f"{name}: {len(df)} total, {n_elong} elongated (1.2–2.0)")

## Orientation histograms

Two-row grid (same style as Exp 3b): top = fitEllipse theta, bottom = angle180.
Filtered to elongated boulders (1.2 ≤ aspect_ratio ≤ 2.0).
Success = flat histogram (no spike at 90° / 175°).

In [ ]:
COLORS = ["tomato", "mediumblue", "darkorange", "darkviolet", "seagreen"]
n_cols = len(estimators)

fig, axes = plt.subplots(2, n_cols, figsize=(5 * n_cols, 10))

for col, ((name, _), color) in enumerate(zip(estimators, COLORS)):
    df = orient_dfs[name]
    df_elong = df[(df['aspect_ratio'] >= 1.2) & (df['aspect_ratio'] <= 2.0)]
    n = len(df_elong)

    axes[0, col].hist(
        df_elong['theta_deg'].dropna(), bins=36, range=(0, 180),
        color=color, edgecolor='k', linewidth=0.4)
    axes[0, col].set_title(f"{name}\nfitEllipse theta (n={n})", fontsize=9)
    axes[0, col].set_xlabel("theta (°)")
    axes[0, col].set_ylabel("count")

    axes[1, col].hist(
        df_elong['angle180'].dropna(), bins=36, range=(0, 180),
        color=color, edgecolor='k', linewidth=0.4)
    axes[1, col].set_title(f"{name}\nangle180 (n={n})", fontsize=9)
    axes[1, col].set_xlabel("angle180 (°)")
    axes[1, col].set_ylabel("count")

plt.suptitle(
    "Orientation experiments 6–9\n"
    "Goal: flatten the 90°/175° spikes present in the Baseline (leftmost column)",
    fontsize=12)
plt.tight_layout()
plt.savefig("exp_orientation_6_to_9.png", dpi=150)
plt.show()

In [ ]:
print(f"{'Method':<35}  {'n_elong':>7}  {'90° spike':>10}  {'175° spike':>11}  {'flat ratio':>11}")
print("-" * 82)

for name, df in orient_dfs.items():
    df_e = df[(df['aspect_ratio'] >= 1.2) & (df['aspect_ratio'] <= 2.0)]
    a180 = df_e['angle180'].dropna()
    n    = len(a180)
    if n == 0:
        print(f"{name:<35}  {'0':>7}  {'—':>10}  {'—':>11}  {'—':>11}")
        continue

    # Spike at 90° (vertical): count in [85°, 95°]
    spike_90  = ((a180 >= 85) & (a180 <= 95)).sum()
    # Spike at 175°: count in [170°, 180°]
    spike_175 = (a180 >= 170).sum()

    # Expected count per 5° bin if flat
    expected_per_5deg = n / 36
    # Flat ratio: actual spike / expected flat — close to 1.0 is good
    flat_90  = spike_90  / expected_per_5deg if expected_per_5deg > 0 else float('nan')

    print(f"{name:<35}  {n:>7}  {spike_90:>10}  {spike_175:>11}  {flat_90:>11.2f}x")

print()
print("flat_90 = count in [85–95°] / expected uniform count per 5° bin")
print("flat_90 ≈ 1.0 means the 90° peak has been eliminated")

## Interpretation notes

### Exp 6 (logit isocontour)
If the SAM2 logit map is itself axis-aligned (i.e., the SAM2 *probability field* has rectilinear structure, not just the hard mask), the isocontour will still be dominated by horizontal/vertical segments and the spikes will persist. If the logit has smooth gradients at the boundary, the contour will be genuinely smooth and the spikes should shrink.

### Exp 7 (Canny edge fitEllipse)
**Convention note:** the `theta_deg` and `angle180` values from `cv2.fitEllipse` may be offset from the baseline by a constant (~90°) depending on how cv2 defines the major axis. The histogram *shape* (spike vs. flat) is the key metric; the absolute positions may need a convention correction.

If the spikes are still present, the image edges *inside* the mask are themselves axis-aligned (e.g., because the boulder boundary in the image is orthogonal to the tile grid — common on Mars where boulders are nearly square). Check with a few qualitative examples.

### Exp 8 (soft moments)
The `theta_deg` / `angle180` values use a direct mathematical convention and may not exactly match the baseline. If the moments give sensible orientations, these histograms should be flat even when the binary mask is rectangular — because the *probability gradient* of the logit rolls off smoothly and is not constrained to be axis-aligned.

### Exp 9 (point prompt)
If the binary mask from the point prompt is still axis-aligned (same visual shape as the box-prompted mask), the histogram will match the baseline. A difference indicates SAM2's mask quality genuinely changes with the prompt type.

---

If **all four approaches still show spikes**, the implication is that the orientation bias is not in the polygon extraction at all — the underlying boulders in this dataset are genuinely preferentially axis-aligned (e.g., due to lighting direction, image registration, or crater ejecta direction). In that case, compare against the ground-truth Prieur shapefiles directly.

In [ ]:
# Optional: compare against Prieur et al. ground-truth orientation
# to determine whether the spikes are real (physical) or artifactual.
import geopandas as gpd

gt_polys = []
for tile_id in gt_tile_ids:
    gt_shp = prieur_test_dir / "labels" / f"M1221383405_{tile_id}_mask.shp"
    if gt_shp.exists():
        gt_polys.append(gpd.read_file(gt_shp))

if gt_polys:
    import warnings
    gdf_gt = gpd.GeoDataFrame(pd.concat(gt_polys, ignore_index=True), crs=gt_polys[0].crs)
    gdf_gt['poly_area'] = gdf_gt.geometry.area
    print(f"Ground-truth boulders loaded: {len(gdf_gt)}")

    gt_records = []
    for _, row in tqdm(gdf_gt.iterrows(), total=len(gdf_gt), desc="GT orientation"):
        try:
            poly    = row.geometry
            row_seg = pd.Series({"geometry": segmentize(poly, tile_res)})
            ellipse_poly, a_fit, b_fit, theta_rad = fitEllipse(row_seg)
            theta_deg = np.degrees(theta_rad)
            mrr_row   = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
            vals      = boulder_row(mrr_row)
            long_axis, short_axis, angle180 = vals[2], vals[3], vals[7]
            ar = long_axis / short_axis if short_axis > 0 else None
            gt_records.append({'theta_deg': theta_deg, 'angle180': angle180,
                                'aspect_ratio': ar, 'poly_area': row['poly_area']})
        except Exception:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                gt_records.append({'theta_deg': None, 'angle180': None,
                                    'aspect_ratio': None, 'poly_area': row['poly_area']})

    df_gt = pd.DataFrame(gt_records)
    df_gt_elong = df_gt[(df_gt['aspect_ratio'] >= 1.2) & (df_gt['aspect_ratio'] <= 2.0)]
    print(f"GT elongated boulders (1.2–2.0): {len(df_gt_elong)}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(df_gt_elong['theta_deg'].dropna(), bins=36, range=(0, 180),
                 color='black', edgecolor='k')
    axes[0].set_title(f"GT (Prieur) fitEllipse theta (n={len(df_gt_elong)})")
    axes[0].set_xlabel("theta (°)")

    axes[1].hist(df_gt_elong['angle180'].dropna(), bins=36, range=(0, 180),
                 color='black', edgecolor='k')
    axes[1].set_title(f"GT (Prieur) angle180 (n={len(df_gt_elong)})")
    axes[1].set_xlabel("angle180 (°)")

    plt.suptitle("Ground-truth orientation: does the Prieur dataset have a real physical bias?")
    plt.tight_layout()
    plt.savefig("exp_orientation_gt_comparison.png", dpi=150)
    plt.show()
else:
    print("No GT shapefiles found — skipping ground-truth comparison.")